In [1]:
import pandas as pd
import re

In [2]:
# ========================== LOAD DATA ==========================
df = pd.read_csv('/Users/tsovinarbabakhanyan/Desktop/Armenian Chat part/src/data/legal_analysis_full_text.csv')

In [3]:
# ========================== LABELING FUNCTION ==========================
def get_decision_label(text):
    if not isinstance(text, str):
        return "Unknown"
    
    text_lower = text.lower()
    text_clean = re.sub(r'\s+', ' ', text)  # clean extra spaces
    
    # Check for final decision section
    if re.search(r'Վ\s*Ճ\s*Ռ\s*Ե\s*Ց|Վճռում է|Վճիռ', text):
        
        # Fully Approved
        if re.search(r'Հայցը բավարարել', text):
            # Check if partial
            if re.search(r'մասնակի|մասով|կարճել|հրաժարվել', text):
                return "Partially Approved"
            return "Fully Approved"
        
        # Rejected
        if re.search(r'Հայցը մերժել', text):
            return "Rejected"
        
        # Divorce cases
        if re.search(r'ամուսնությունը լուծել|ամուսնալուծել', text) and re.search(r'բավարարել', text):
            return "Approved (Divorce)"
        
        # Contract termination + compensation
        if re.search(r'պայմանագիրը լուծել|բռնագանձել', text):
            return "Fully Approved"
    
    # Partial withdrawal or case closed
    if re.search(r'հրաժարվել.*պահանջ|վարույթը կարճել', text_lower):
        return "Partially Approved / Closed"
    
    # Default fallback
    if "բավարարել" in text_lower:
        return "Approved"
    if "մերժել" in text_lower:
        return "Rejected"
    
    return "Unknown / Needs Manual Check"


In [6]:
# ========================== APPLY LABEL ==========================
df['Decision_Label'] = df['Full_Document_Text'].apply(get_decision_label)

# ========================== RESULTS ==========================
print("Label Distribution:")
print(df['Decision_Label'].value_counts())

# Show full table
print("\n=== LABELED RESULTS ===")
print(df[['Case_Number', 'Parties', 'Claim_Type', 'Decision_Label']])

# Save to new CSV
df.to_csv('/Users/tsovinarbabakhanyan/Desktop/Armenian Chat part/src/data/legal_analysis_labeled.csv', index=False, encoding='utf-8')
print("\n✅ Saved as 'legal_analysis_labeled.csv'")

Label Distribution:
Decision_Label
Partially Approved    5
Name: count, dtype: int64

=== LABELED RESULTS ===
       Case_Number                                            Parties  \
0   ԵԴ2/3094/02/24  Աննա Համլետի\nԱվետիսյանը vs Հակոբ Արազ Նավասար...   
1   ԵԴ2/2355/02/24  Արթուր Սարգսի Սարգսյանի vs Գլոբալ Շիփփինգ ՍՊ ը...   
2   ԵԴ2/8652/02/25  Գայանե Վարդազարի Բեգլարյանի vs Վլադիմիր Վարդազարի   
3  ԵԴ2/13767/02/25  Անժելա Գագիկի Մառանջյանի vs Վարդան Գևորգի Մառա...   
4  ԱՎԴ1/0074/02/24  Արմեն Ալբերտի Վարդանյանը vs Դիանա Հայկի Մաթևոս...   

                                          Claim_Type      Decision_Label  
0  հողի վարձակալության պայմանագիրը լուծելու պահան...  Partially Approved  
1                 գումարի բռնագանձման  պահանջի մասին  Partially Approved  
2  Դատարան) ընդդեմ Սուրեն Խաչատուրի Փիրջանյանի՝ գ...  Partially Approved  
3                  գումարի բռնագանձման պահանջի մասին  Partially Approved  
4  Արման Հովիկի Գրիգորյանին անգործունակ ճանաչելու...  Partially Approved  

